## Decision log (what this notebook implements)

| #   | Decision                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       |
| --- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| D1  | US locale only (ESCI also has `es`, `jp`; DistilBERT pipeline is English)                                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| D2  | ESCI has no category field → electronics is inferred from text (keywords + vetoes)                                                                                                                                                                                                                                                                                                                                                                                                                                             |
| D3  | Match on brand + title with word boundaries; bullets/description are not used (they list compatible devices)                                                                                                                                                                                                                                                                                                                                                                                                                   |
| D4  | Keyword list = the 5 original inclusion groups + 3 groups added in v2.2: `computer_peripherals` (monitors, keyboards, mice, printers), `gaming` (game controllers), `smart_home` (smart plugs/bulbs/switches); microphones → `audio`, camera lenses → `imaging`. RAM context-aware, routers need network context, laptop product lines added                                                                                                                                                                                   |
| D5  | Brand alone never qualifies. Brand + model token qualifies; Samsung/Sony/LG/Panasonic/Philips/Sharp/Toshiba additionally need a hardware noun                                                                                                                                                                                                                                                                                                                                                                                  |
| D6  | Vetoes: accessory head nouns (query-only strict forms), homonyms, outdoor/power tools, woodworking routers, furniture, exercise gear, fashion/novelty, phone plans/SIM kits, optics, books, case brands, replacement parts, consumables (ink, paper, filament), new-scope guards (baby/medical monitors, piano keyboards, mouse pests/pads, contact lenses). Remaining out-of-scope buckets: game consoles (game titles carry console names), projectors, drones, e-readers, scanners — each re-admittable via `allow_buckets` |
| D7  | Query-level decision from the query's Exact/Substitute products; keep the query's full judged list (ESCI's original design: \~20 judged results per query)                                                                                                                                                                                                                                                                                                                                                                     |
| D8  | Preserve ESCI metadata: `split`, `small_version`, `large_version`, `source`                                                                                                                                                                                                                                                                                                                                                                                                                                                    |
| D9  | NDCG gains = ESCI paper values E=1.0, S=0.1, C=0.01, I=0.0 (proposal currently says 1.0/0.5/0.1/0.0)                                                                                                                                                                                                                                                                                                                                                                                                                           |
| D10 | Classifier pool: sources `other` + `parse_pattern`, must have ≥1 E/S product, length buckets 90/230/110/70, off-topic queries labelled `N/A`                                                                                                                                                                                                                                                                                                                                                                                   |
| D11 | Classifier queries drawn from ESCI train split only; test split untouched until the final ablation                                                                                                                                                                                                                                                                                                                                                                                                                             |
| D13 | v2.3 is the final patch and is frozen (version-tagged, thresholds a = 0.50, b = 0.80, n_min = 3). Nothing is patched after the audit; residual errors are reported, not fixed                                                                                                                                                                                                                                                                                                                                                  |
| D12 | Re-audit at query level on a fresh, blind, stratified sample (seed 2026, \~310 queries, excluding every query seen during development): stratified estimators for precision and recall with bootstrap 95% CIs                                                                                                                                                                                                                                                                                                                  |

## Step 1 - Load ESCI (US only)

In [2]:
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path
import html

DATA = "../data/"
OUT  = Path(DATA) / "processed"          # every output of this notebook goes here
OUT.mkdir(parents=True, exist_ok=True)

sources  = pd.read_csv(DATA + "shopping_queries_dataset_sources.csv")
products = pd.read_parquet(DATA + "shopping_queries_dataset_products.parquet")
examples = pd.read_parquet(DATA + "shopping_queries_dataset_examples.parquet")

us_products = products[products["product_locale"] == "us"].copy()
us_examples = examples[examples["product_locale"] == "us"].copy()

check = us_examples.groupby("query_id")["small_version"].nunique()
bad_ids = check.index[(check > 1)]
pd.Series(bad_ids, name="query_id").to_csv(OUT / "excluded_inconsistent_queries.csv", index=False)
us_examples = us_examples[~us_examples["query_id"].isin(bad_ids)].copy()

# one row per US query: ESCI split / version flags / number of judged products
q_raw = us_examples.groupby("query_id").agg(
    n_judged=("example_id", "size"),
    split=("split", "first"),
    small=("small_version", "max"),
    large=("large_version", "max"),
)

## Step 2 - Filter definitions (v2.3, frozen)
Only the word lists and compiled patterns. They are used directly in Step 3.

In [3]:
# 1. CATEGORY TERMS
CATEGORY_TERMS = {
    "mobile_computing": [
        r"smart ?phones?", r"cell ?phones?", r"mobile phones?", r"iphones?", r"phones?",
        r"galaxy (?:s|a|z|m|note)\s?\d+\w*", r"pixel \d\w*", r"redmi", r"poco [a-z]\d+", r"oneplus", r"moto [a-z]\d*",
        r"laptops?", r"notebook (?:computers?|pcs?)",
        r"ideapad", r"thinkpad", r"thinkbook", r"vivobook", r"zenbook", r"inspiron", r"latitude", r"alienware",
        r"elitebook", r"probook", r"surface (?:pro|go|laptop)", r"galaxy book", r"chromebooks?", r"macbooks?", r"ultrabooks?",
        r"ipads?", r"galaxy tab\w*", r"fire (?:hd |\d{1,2} )?(?:kids )?tablets?",
        r"(?:android|windows|kids?|drawing|graphics?|pen|digital|hd|wi-?fi|lte|samsung|lenovo|amazon|huawei|apple|tcl|\d{1,2}(?:\.\d)?[- ]?(?:inch|in)) tablets?",
        r"tablets? (?:pc|computer|\d{1,2}(?:\.\d)? ?(?:inch|in)|for (?:kids|drawing|reading|gaming|seniors|toddlers|students|work))",
        r"(?:desktop|gaming|mini|all[- ]in[- ]one) (?:computers?|pcs?)",
        r"smart ?watch(?:es)?", r"apple watch", r"galaxy watch", r"fitbit", r"amazfit",
        r"fitness (?:trackers?|bands?)", r"garmin (?:forerunner|fenix|venu|vivoactive)",
    ],
    "components_accessories": [
        # RAM needs memory context (bare "ram" = Dodge Ram, RAM Golf, "ram" in other words)
        r"\d+\s?(?:gb|tb|mb)\s*(?:\w+\s+){0,2}ram", r"ram\s*\d+\s?(?:gb|tb|mb)",
        r"(?:laptop|desktop|computer|memory|pc|gaming|server) ram", r"ram (?:memory|kit|modules?|upgrade)s?",
        r"ddr[2-5]", r"so-?dimm", r"dimm",
        r"ssds?", r"solid[- ]state drives?", r"nvme", r"m\.2", r"hdds?", r"hard (?:disk )?drives?", r"external (?:hard )?drives?",
        r"(?:usb |flash |thumb |pen )drives?", r"micro ?sd(?:hc|xc)?(?: cards?)?", r"sd cards?",
        r"memory cards?", r"card readers?",
        r"cpus?", r"processors?", r"ryzen", r"intel core", r"core i[3579]", r"threadripper", r"xeon",
        r"gpus?", r"graphics cards?", r"video cards?", r"geforce", r"rtx ?\d{3,4}", r"gtx ?\d{3,4}",
        r"radeon", r"motherboards?", r"psu", r"(?:pc|computer|cpu|case|radiator) fans?",
        r"power ?banks?", r"portable chargers?", r"chargers?", r"charging (?:cables?|docks?|stations?)",
        r"usb[- ]?c", r"usb (?:type[- ]c |micro |mini )?cables?", r"micro usb", r"hdmi",
        r"displayport", r"lightning (?:cables?|chargers?)", r"ethernet cables?", r"cat ?[5-8]e? cables?",
        r"aux cables?", r"airtags?", r"usb hubs?", r"docking stations?", r"usb (?:adapters?|extension)",
    ],
    "imaging": [
        r"cameras?", r"dash ?cams?", r"web ?cams?", r"camcorders?", r"gopro", r"dslr", r"mirrorless",
        r"action cam(?:era)?s?", r"cctv", r"ip cameras?", r"video doorbells?", r"doorbell cam(?:era)?s?", r"ring doorbell", r"smart doorbell", r"nanny cams?",
        r"trail cameras?", r"instax", r"lens(?:es)?",                          # NEW scope: camera lenses
    ],
    "home_electronics": [
        r"tvs?", r"televisions?", r"fire tv(?: stick| cube)?", r"roku", r"chromecast",
        r"streaming (?:devices?|sticks?)", r"tv box", r"soundbars?", r"echo (?:dot|show|pop|studio)",
        r"homepod", r"google nest (?:mini|audio|hub|wi-?fi)",
        r"(?:bluetooth|smart|portable|wireless|computer|pc|party|waterproof|bookshelf|usb|home theater) speakers?",
        r"speakers?",
        # routers only with network context (bare "router" = woodworking tool)
        r"(?:wi-?fi|wireless|mesh|gaming|dual[- ]band|tri[- ]band|[345]g|lte|modem|gigabit|ethernet|vpn|smart) routers?",
        r"routers? (?:ac|ax)\d{3,4}", r"(?:wi-?fi|range|signal) extenders?", r"mesh wi-?fi", r"wi-?fi (?:system|6|7)",
        r"modems?", r"access points?", r"network switch(?:es)?", r"powerline adapters?",
        r"(?:desk|table|tower|pedestal|standing|stand|floor|usb|clip[- ]on|oscillating|rechargeable|"
        r"portable|box|window|bladeless|misting|personal|neck|battery[- ](?:operated|powered)) fans?",
        r"air circulators?",
    ],
    "audio": [
        r"head ?phones?", r"ear ?buds?", r"ear ?phones?", r"airpods?", r"headsets?", r"in-?ear",
        r"over-?ear", r"on-?ear", r"true wireless", r"tws", r"neckbands?", r"bone conduction",
        r"microphones?", r"mics?", r"lavalier",                                  # NEW scope: microphones
    ],
    # ---- NEW in scope (v2.2): monitors, keyboards, mice, printers ----
    "computer_peripherals": [
        r"monitors?", r"keyboards?", r"mice", r"mouse (?:for|with) (?:laptop|pc|computer|mac|ipad)",
        r"(?:computer|gaming|wireless|usb|bluetooth|optical|ergonomic|vertical|laptop|pc) mouse",
        r"printers?",
    ],
    # ---- NEW in scope: game controllers (consoles themselves stay OUT: game titles carry console names) ----
    "gaming": [
        r"game ?controllers?", r"gamepads?", r"joysticks?", r"joy-?cons?",
        r"(?:ps[2345]|xbox(?: one| 360| series [xs])?|switch|nintendo(?: switch)?|wii(?: u)?|playstation ?\d?|pc|gaming|"
        r"wireless|bluetooth|usb|pro|steam|android|phone|mobile) controllers?",
        r"controllers? for (?:gaming|games|phone|android|iphone|pc)",
    ],
    # ---- NEW in scope: smart plugs / bulbs / switches ----
    "smart_home": [
        r"smart (?:plugs?|bulbs?|switch(?:es)?|outlets?|lights?|lighting)", r"(?:wi-?fi|wifi|zigbee) (?:plugs?|outlets?|switch(?:es)?|bulbs?)",
        r"smart light switch(?:es)?", r"zigbee", r"z-wave", r"philips hue", r"\bkasa\b", r"wemo", r"lifx",
    ],
}

# every group becomes one pattern; the first group that matches names the text (dict order matters)
CATEGORY_RX = {g: re.compile(r"\b(?:" + "|".join(p) + r")\b", re.I) for g, p in CATEGORY_TERMS.items()}

# Devices that remain OUT of scope by default. Each can be re-admitted with allow_buckets={...}.
# NOTE "consoles": console names are part of video-game titles ("Mario Kart - Nintendo Switch"), so admitting
# them floods the dataset with software. Controllers/headsets for consoles are handled by the in-scope groups.
OOS_BUCKETS = {
    "consoles":   [r"ps[2345]", r"playstation(?: ?\d)?", r"xbox(?: one| 360| series [xs])?", r"nintendo(?: switch)?",
                   r"wii(?: u)?", r"game consoles?", r"steam deck"],
    "projectors": [r"projectors?"],
    "drones":     [r"drones?", r"quadcopters?"],
    "ereaders":   [r"e-?readers?", r"kindle(?! fire)"],
    "scanners":   [r"scanners?"],
}
OOS_RX = {b: re.compile(r"(?<!with\s)(?<!without\s)(?<!for\s)(?<!and\s)(?<!w/)\b(?:" + "|".join(p) + r")\b", re.I)
          for b, p in OOS_BUCKETS.items()}
OOS_RX = {b: re.compile(r"(?<!with\s)(?<!without\s)(?<!for\s)(?<!and\s)(?<!w/)\b(?:" + "|".join(p) + r")\b", re.I)
          for b, p in OOS_BUCKETS.items()}

# ---------------- 2. ACCESSORY VETO (accessory is the head noun) ----------------
_ACC = (r"cases?(?!\s+fans?)|covers?|screen protectors?|protectors?|tempered glass|skins?|sleeves?|"
        r"pouch(?:es)?|holsters?|bags?|backpacks?|stickers?|decals?|tripods?|lanyards?|wall mounts?|"
        r"mounts?|brackets?|stands?|holders?|straps?|grips?|lens caps?|remote covers?|"
        r"desks?|tables?|carts?|shelf|shelves|chairs?|risers?|remotes?|remote controls?")
ACC_TAIL = re.compile(rf"(?<!with\s)(?<!without\s)(?<!no\s)(?<!and\s)(?<!w/)(?<!charging\s)(?<!for\s)(?<!-)(?<!\be\s)(?<!\brf\s)(?<!\bz\s)(?<!\bf\s)(?<!\bx\s)(?<!\bef\s)(?<!\bl\s)\b(?:{_ACC})(?:\s+[\d.]+(?:\s*(?:pack|pcs|ct|count|inch|in))?)?\s*$", re.I)
ACC_FOR = re.compile(rf"\b(?:{_ACC})\b[^,]{{0,40}}?\b(?:for|compatible with|fits?)\b", re.I)
WATCH_BAND = re.compile(r"\bwatch\s+(?:bands?|straps?)\b|\bbands? for (?:apple |galaxy |smart )?watch", re.I)

# ---------------- 3. HARD VETO (homonyms + the failure classes found in audits) ----------------
HARD_VETO_TERMS = [
    # original homonyms
    r"food processors?", r"word processors?", r"dodge", r"dodge ram", r"ram (?:\d{3,4}|trucks?|pickup|hitch|promaster|horns?)",
    r"charger challenger", r"as seen on tv", r"tv (?:shows?|series|guide|trays?|dinners?|cabinet)",
    r"writing tablets?", r"emerald tablet", r"fan (?:fiction|fest|club|gear|shop)",
    r"ceiling fans?", r"exhaust fans?", r"hand ?held manual", r"hand fans?", r"folding fans?", r"paper fans?",
    r"vitamins?", r"supplements?", r"capsules?", r"chewables?", r"effervescent", r"melatonin", r"probiotics?",
    r"denture", r"water purification", r"chlorine", r"iodine", r"toys?", r"pretend play", r"dolls?", r"costumes?",
    r"ink", r"toners?", r"cartridges?", r"car battery", r"trickle", r"jump starters?",
    # large appliances
    r"washing machines?", r"washers?", r"dryers?", r"refrigerators?", r"fridges?", r"freezers?", r"microwaves?",
    r"ovens?", r"range hoods?", r"air conditioners?", r"vacuum(?: cleaners?)?", r"cooktops?", r"rice cookers?",
    r"air fryers?", r"blenders?", r"dishwasher", r"detergent",
    # audit class: woodworking / outdoor & power tools
    r"router (?:bits?|tables?|lifts?|bases?|collets?|jigs?|planes?|guides?|templates?|sleds?|depth gauges?|replacement)",
    r"cnc routers?", r"wood(?:en|working)? routers?", r"trim routers?", r"plunge routers?", r"laminate trimmers?",
    r"edgers?", r"trimmers?", r"chainsaws?", r"lawn ?mowers?", r"leaf ?blowers?", r"pole ?saws?", r"weed ?wackers?",
    r"pressure washers?", r"power tools?", r"cordless drills?", r"impact drivers?", r"circular saws?", r"jig ?saws?",
    r"angle grinders?",
    # audit class: furniture
    r"(?:computer|laptop|standing|gaming|writing|office|folding) desks?", r"tv stands?", r"entertainment centers?",
    r"media consoles?", r"credenzas?", r"bookcases?", r"dressers?", r"nightstands?", r"sofas?", r"couch",
    # audit class: exercise equipment
    r"exercise bikes?", r"stationary bikes?", r"spin bikes?", r"treadmills?", r"ellipticals?", r"rowing machines?",
    # audit class: fashion / novelty
    r"crossbody", r"wallets?", r"handbags?", r"totes?", r"drawstring", r"leggings", r"t-?shirts?",
    r"hoodies?", r"socks", r"charm straps?", r"phone charms?", r"jewelry",
    # audit class: services / plans, optics, books
    r"sim kits?", r"unlimited talk", r"prepaid (?:plans?|cards?|sim|data|minutes|service|wireless|airtime)", r"data plans?", r"activation kits?", r"wireless (?:service|plans?)",
    r"telescopes?", r"monoculars?", r"binoculars?", r"microscopes?", r"spotting scopes?",
    r"a novel", r"a memoir", r"paperback", r"hardcover", r"audiobooks?",
    # v2.1: case/protector brands, collectibles, tripods, replacement parts, pharma leaks
    r"otterbox", r"spigen", r"ringke", r"supcase", r"caseology", r"zagg", r"urban armor gear", r"lifeproof",
    r"incipio", r"casetify", r"ghostek", r"tech21", r"funko", r"figurines?", r"action figures?", r"collectibles?",
    r"tripods?", r"selfie sticks?", r"monopods?", r"(?:battery|screen|lcd|digitizer|display) replacement",
    r"replacement (?:batter(?:y|ies)|screens?|parts?|lcd)", r"repair (?:kits?|tools?)",
    r"mg", r"allergy", r"antihistamine", r"ibuprofen", r"acetaminophen", r"aspirin", r"antacid", r"medicine",
    r"acid reducer", r"niacinamide", r"loratadine", r"cetirizine", r"multivitamins?", r"biotin", r"zinc",
    # v2.2: more case brands, parts, consumables, lighting accessories
    r"speck", r"pelican", r"rhinoshield", r"nillkin", r"case-?mate", r"poetic", r"ulak", r"miracase", r"zizo",
    r"tudia", r"fintie", r"procase", r"jetech", r"smartish",
    r"battery (?:cover|door)s?", r"back (?:cover|glass)", r"glass back", r"housing", r"flashlights?", r"ring lights?",
    r"selfie lights?", r"instax[\w ]{0,12}films?", r"(?:printer|photo|copy|inkjet|glossy|label|sticker) paper",
    r"filaments?", r"printer (?:ink|ribbons?|labels?|cartridges?|toners?|rolls?)",
    # new-scope false-positive guards: pests, medical/baby monitors, musical keyboards, mouse pads, lens eyewear
    r"mouse (?:traps?|pads?|ears?|mat)", r"mice (?:traps?|repellent)", r"(?:rat|mouse|mice) (?:repellent|poison|bait|control)",
    r"mickey mouse", r"minnie mouse", r"mousepads?",
    r"(?:baby|heart rate|blood pressure|glucose|fetal|sleep|pulse|breathing) monitors?",
    r"monitor (?:arms?|lights?|light bars?|risers?|stands?|mounts?|hoods?|shades?)",
    r"(?:piano|music|musical|midi|casio|yamaha) keyboards?", r"keyboard (?:stands?|benches?|bags?|covers?|cats?)",
    r"contact lens(?:es)?", r"eyeglass(?:es)?", r"sunglass(?:es)?", r"reading glasses", r"prescription", r"magnifying",
    r"lens (?:cloths?|wipes?|cleaners?|pens?|cleaning|hoods?|filters?|caps?|adapter rings?|cases?)",
    r"switch (?:plates?|covers?)", r"outlet covers?", r"wall plates?",
    r"tpms", r"doppler", r"fetal", r"heartbeat monitors?",
    r"(?:tire|tyre|pressure|temperature|humidity|air quality|co2|radon|energy|pet|dog|cat|bed|wet|moisture|aquarium|fish tank) monitors?",
    r"microphone (?:stands?|arms?)", r"mic (?:stands?|arms?)", r"pop filters?", r"windscreens?", r"boom arms?",
    r"(?:ps[2345]|xbox(?: one| 360)?|switch|nintendo(?: switch)?|wii|video|board|card|party) games?(?!\s*(?:controllers?|consoles?|pads?|sticks?))",
]
HARD_VETO = re.compile(r"\b(?:" + "|".join(HARD_VETO_TERMS) + r")\b", re.I)

# ---- QUERY-ONLY vetoes (too aggressive for product titles: "AirPods Pro with MagSafe Charging Case",
#      "Fitbit Charge 4 (S & L Bands Included)", "Laptop ... Backlit Keyboard" are real electronics)
CASE_ANY = re.compile(r"(?<!charging\s)(?<!without\s)(?<!no\s)\bcases?\b", re.I)
PC_CASE_OK = re.compile(r"\b(?:pc|computer|gaming|tower|atx|itx|server|open|mid[- ]tower|full[- ]tower)\s+cases?\b|"
                        r"\bcases?\s+fans?\b|\bcases? (?:for|with) (?:a )?(?:gaming )?(?:pc|computer)\b", re.I)
BAG_ANY = re.compile(r"\b(?:backpacks?|bags?|sleeves?|briefcases?|messenger|luggage)\b", re.I)
DEV_ACC = re.compile(rf"\b(?:phones?|iphones?|smart ?phones?|laptops?|tablets?|ipads?|cameras?|tvs?|airpods?|watch(?:es)?|macbooks?)\s+(?:{_ACC})\b", re.I)
WATCH_BAND2 = re.compile(r"\b(?:watch(?:es)?|fitbit|garmin|amazfit|versa|alta|charge \d|inspire|forerunner|vivoactive|vivosmart)\b[^,]{0,40}?(?<!without\s)(?<!no\s)\b(?:bands?|straps?|wristbands?)\b", re.I)

## Step 3 - Apply the frozen v2.3 filter to ALL US queries
Thresholds are frozen:  
- Rule A share ≥ 0.50 (query has an electronics term).
- Rule B share ≥ 0.80 and ≥ 3 E/S products (no term).

A product counts as electronic when its title is not vetoed and matches a category group.  
A query is vetoed by the hard / accessory / out-of-scope rules.

In [4]:
FILTER_VERSION = "v2.3"
A_CUT, B_CUT, N_MIN = 0.50, 0.80, 3

# 3A. product level: veto first, then the first matching(next get the first match) category group (the brand + model route is OFF)
title = us_products["product_title"].fillna("").tolist()

prod_group = [
    None if (
        HARD_VETO.search(t) 
        or ACC_TAIL.search(t) 
        or ACC_FOR.search(t) 
        or WATCH_BAND.search(t)
    )
    else next(
        (g for g, rx in CATEGORY_RX.items() if rx.search(t)), 
        None
    )
    for t in title
]

prod = pd.DataFrame({
    "product_id": us_products["product_id"].to_numpy(), 
    "prod_group": prod_group
})

prod["prod_elec"] = prod["prod_group"].notna()

#DF now has product_id, prod_group, prod_elec

# 3B. query level: electronics share among each query's Exact / Substitute products

ex = us_examples[
    ["query_id", "product_id", "esci_label"]
].merge(prod, on="product_id", how="left")

ex["prod_elec"] = ex["prod_elec"].fillna(False).astype(bool)

es = ex[ex["esci_label"].isin(["E", "S"])]

g_es = es.groupby("query_id").agg(
    n_es=("prod_elec", "size"), 
    share_es=("prod_elec", "mean")
)
# Find the most common electronics category for each query
top  = (
    es[es["prod_elec"]]
    .groupby("query_id")["prod_group"]
    .agg(lambda s: s.mode().iat[0])
    .rename("top_prod_group")
)

q = (us_examples.drop_duplicates("query_id")[["query_id", "query"]].set_index("query_id")
     .join(q_raw[["split", "small", "n_judged"]])
     .join(sources.set_index("query_id"))
     .join(g_es).join(top))

# 3C. query text: veto reason (hard > accessory > out-of-scope bucket) and electronics group
veto_reason, query_group = [], []
for t in q["query"]:
    if HARD_VETO.search(t):
        r = "hard"
    elif (ACC_TAIL.search(t) or ACC_FOR.search(t) or WATCH_BAND.search(t)
          or (CASE_ANY.search(t) and not PC_CASE_OK.search(t))
          or DEV_ACC.search(t) or WATCH_BAND2.search(t) or BAG_ANY.search(t)):
        r = "accessory"
    else:
        r = None
        if len(t.split()) <= 6:  # long natural-language queries mention peripherals as requirements, not as the product
            for b, rx in OOS_RX.items():
                # "xbox controller usb cable" stays in scope
                if rx.search(t) and not (b == "consoles" and CATEGORY_RX["gaming"].search(t)):
                    r = "oos:" + b
                    break
    veto_reason.append(r)
    query_group.append(None if r else next((g for g, rx in CATEGORY_RX.items() if rx.search(t)), None))

q["veto_reason"]  = veto_reason
q["query_group"]  = query_group
q["query_vetoed"] = q["veto_reason"].notna()

q["n_es"] = q["n_es"].fillna(0).astype(int)
q["share_es"] = q["share_es"].fillna(0.0)

# 3D. keep rules
ok = ~q["query_vetoed"]
q["rule_A"] = ok & q["query_group"].notna() & (q["share_es"] >= A_CUT)

q["rule_B"] = ok & q["query_group"].isna() & (q["share_es"] >= B_CUT) & (q["n_es"] >= N_MIN)

q["v2_keep"] = q["rule_A"] | q["rule_B"]

q["scope_group"] = q["query_group"].fillna(q["top_prod_group"])

q.assign(filter_version=FILTER_VERSION).to_csv(OUT / f"query_table_{FILTER_VERSION}.csv")

print(FILTER_VERSION, "frozen:", int(q["v2_keep"].sum()), "of", len(q), "US queries kept | Rule A", int(q["rule_A"].sum()), "| Rule B", int(q["rule_B"].sum()))

v2.3 frozen: 5664 of 97338 US queries kept | Rule A 4188 | Rule B 1476


## Step 4 - Audit protocol for the frozen v2.3 filter
**Protocol (do not deviate):** 
- (1) the filter and thresholds are frozen, nothing is patched after this point;
- (2) the audit sample is drawn with a new seed and **excludes every query that appeared in a development table**;
- (3) label **blind** (the file shows only the query and example product titles, not the filter's decision);
- (4) precision and recall are computed with stratified estimators and intervals;
- (5) residual errors are *reported*, not fixed.

In [5]:
# 4A. stratified audit sample (seed 2026) from queries NOT seen during development
# every query id that was displayed in a development table
DEV_SEEN_IDS = {833, 1203, 1309, 1505, 1743, 1747, 1792, 2079, 2382, 2693, 2703, 3022, 3034, 3141, 3355, 3496, 3678, 4222, 4348, 4630, 5046, 5060, 5735, 5750, 6847, 7483, 7492, 7522, 7540, 7565, 7583, 7586, 7587, 7687, 7883, 7901, 8027, 8369, 8729, 9059, 9267, 9692, 9785, 9811, 9848, 9921, 9922, 10297, 10436, 10839, 10924, 11259, 11521, 11602, 11913, 12075, 12433, 12915, 13907, 14380, 14445, 15934, 17011, 17012, 17605, 17889, 18381, 18974, 19294, 19547, 20878, 21182, 21885, 21895, 21906, 22391, 22469, 22494, 22495, 22729, 22742, 23393, 23571, 23793, 23928, 24947, 26902, 27844, 28355, 28363, 28365, 28377, 28403, 28720, 29226, 29253, 29259, 29260, 31032, 31823, 32048, 32162, 32239, 32542, 32606, 32816, 33021, 33227, 33554, 33875, 34657, 34884, 35601, 36129, 36628, 36960, 37206, 37643, 37884, 38741, 39536, 39547, 40454, 40706, 40796, 40815, 40818, 40821, 40824, 40829, 40830, 40832, 40833, 40835, 40864, 40888, 40897, 41295, 42079, 42124, 42720, 42933, 42991, 43013, 43019, 43969, 44005, 44014, 44026, 44067, 44072, 44251, 44271, 44273, 44294, 44497, 44504, 44519, 44704, 45310, 45407, 46036, 46958, 47249, 48101, 48254, 49550, 49738, 49778, 50536, 51167, 51175, 51183, 51686, 51726, 51747, 51854, 51876, 52193, 52310, 52574, 53850, 53872, 53993, 54024, 54055, 54066, 54112, 54158, 54198, 54213, 54233, 54490, 54505, 54510, 54593, 56378, 56748, 56997, 57448, 57466, 57482, 57501, 57573, 57703, 57721, 57807, 58301, 58617, 59021, 59179, 59955, 60132, 60222, 61129, 61321, 61502, 61777, 61927, 63294, 63307, 63679, 64523, 66766, 67097, 68276, 68644, 68903, 69436, 69913, 69937, 70386, 70864, 70868, 70881, 70975, 71083, 71205, 71297, 71335, 71697, 71704, 72309, 72982, 73248, 73332, 73866, 73879, 73893, 74158, 74272, 74456, 74480, 74495, 74499, 74617, 74668, 75531, 75704, 75796, 75800, 75849, 77173, 78301, 78338, 78347, 78486, 78635, 79434, 79456, 79572, 79579, 79595, 79602, 79722, 80611, 81667, 81795, 81833, 81926, 81929, 82417, 82483, 82806, 82884, 82937, 83134, 83412, 83882, 83940, 84651, 84801, 84844, 85467, 85787, 85803, 85813, 86185, 87745, 88117, 88801, 89112, 89406, 89648, 89956, 89967, 89991, 89996, 90010, 90053, 90123, 90152, 90200, 90231, 90232, 90352, 91153, 91320, 91712, 92138, 93170, 94003, 94615, 94992, 94993, 95017, 95047, 95493, 96071, 96089, 96162, 96575, 96581, 96582, 96623, 96748, 96807, 96809, 97507, 97975, 98507, 100920, 100923, 101780, 101948, 102260, 103017, 103458, 104392, 104471, 104473, 104592, 104596, 105001, 105305, 105482, 105890, 105944, 106235, 106333, 106438, 107195, 107293, 107303, 107363, 107392, 108564, 108809, 109052, 109164, 109211, 109831, 110076, 111141, 111636, 111638, 111814, 111816, 112109, 112160, 112163, 113974, 114206, 114239, 114250, 114355, 114606, 115022, 115695}

frame = q[~q.index.isin(DEV_SEEN_IDS)].copy()

oos = frame["veto_reason"].fillna("").str.startswith("oos:")
has_term = frame["query_group"].notna()
unv = ~frame["query_vetoed"]

conds = [
    frame["rule_A"],
    frame["rule_B"],
    frame["query_vetoed"] & ~oos & (frame["n_es"] >= 3) & (frame["share_es"] >= 0.5),
    frame["query_vetoed"],
    unv & ~frame["v2_keep"] & ((has_term & frame["share_es"].between(0.3, 0.4999)) |
                               (~has_term & (frame["n_es"] >= 3) & frame["share_es"].between(0.5, 0.7999))),
    unv & ~frame["v2_keep"] & (frame["share_es"] >= 0.1),
    unv & ~frame["v2_keep"],
]

names = ["K_A_ruleA", "K_B_ruleB", "D1_vetoed_high_share", "D5_vetoed_other", "D2_near_miss", "D3_low_share", "D4_bulk"]

frame["stratum"] = np.select(
    conds, 
    names, 
    default="other"
)

SAMPLE_N = {"K_A_ruleA": 70, "K_B_ruleB": 50, "D1_vetoed_high_share": 40, "D5_vetoed_other": 30,
            "D2_near_miss": 50, "D3_low_share": 30, "D4_bulk": 40}

rng = np.random.default_rng(2026)
pick = []

for st, n in SAMPLE_N.items():
    ids = frame.index[frame["stratum"] == st].to_numpy()
    pick += list(rng.choice(ids, size=min(n, len(ids)), replace=False))
sample = frame.loc[pick].copy()

title_of = us_products.set_index("product_id")["product_title"]

rows = us_examples[us_examples["query_id"].isin(sample.index) & us_examples["esci_label"].isin(["E", "S"])]

rows = rows.sort_values(["query_id", "esci_label"])
ex_titles = rows.groupby("query_id")["product_id"].apply(lambda s: " || ".join(title_of.reindex(s.iloc[:3]).fillna("").str.slice(0, 90)))
sample["example_titles"] = ex_titles.reindex(sample.index).fillna("")

blind = sample.reset_index().sample(frac=1, random_state=7).reset_index(drop=True)      # shuffled
blind["audit_id"] = np.arange(1, len(blind) + 1)
blind[["audit_id", "query", "example_titles"]].assign(in_scope="", note="").to_csv(OUT / "audit_to_label.csv", index=False)
blind[["audit_id", "query_id", "stratum", "v2_keep", "rule_A", "rule_B", "veto_reason", "n_es", "share_es"]].to_csv(OUT / "audit_key.csv", index=False)
sizes = pd.concat([frame["stratum"].value_counts().rename("N_h"), sample["stratum"].value_counts().rename("n_h")], axis=1)
sizes.index.name = "stratum"
sizes.to_csv(OUT / "audit_strata_sizes.csv")
print(sizes)

                        N_h  n_h
stratum                         
D4_bulk               72472   40
D5_vetoed_other       14601   30
K_A_ruleA              4038   70
D3_low_share           3080   30
K_B_ruleB              1407   50
D2_near_miss           1251   50
D1_vetoed_high_share    100   40


### Labeling guide (apply literally; label **blind**, in the order given)
Open `audit_to_label.csv`, fill the `in_scope` column, save as **`audit_labeled.csv`**.

* **1** = the shopper's main intent is to buy a **device or component** in the project scope: smartphones, laptops, tablets, desktops, smartwatches/fitness trackers · RAM, storage, CPUs/GPUs/motherboards, chargers, cables, power banks, USB hubs/adapters · cameras (incl. lenses, dash/web/security/action cams) · TVs, speakers/soundbars, routers/modems/extenders, electric fans · headphones/earbuds/headsets/microphones · monitors, keyboards, mice, printers · game controllers · smart plugs/bulbs/switches.
* **0** = anything else: accessories-only (cases, covers, screen protectors, bands/straps, stands, mounts, bags, tripods), replacement parts, consumables (ink, paper, film, filament), software/games/apps, phone plans, large appliances, tools, toys, furniture, books, or non-electronics.
* **u** = genuinely cannot decide (reported separately, excluded from the estimates).

In [6]:
# 4B. RUN AFTER LABELING -> precision / recall (stratified estimators)
lab = pd.read_csv(OUT / "audit_labeled.csv", dtype=str, keep_default_na=False)
lab["audit_id"] = lab["audit_id"].astype(int)
key = pd.read_csv(OUT / "audit_key.csv")
sz = pd.read_csv(OUT / "audit_strata_sizes.csv", index_col=0)

d0 = lab[["audit_id", "in_scope", "query"]].merge(key, on="audit_id")
d0["y"] = d0["in_scope"].astype(str).str.strip().str.lower().map({"1": 1, "0": 0, "1.0": 1, "0.0": 0})
print("unsure / unlabeled:", int(d0["y"].isna().sum()), "of", len(d0))

d = d0.dropna(subset=["y"]) #remove unlabeled once

per = d.groupby("stratum")["y"].agg(
    k="sum", # has only 1,0 so sum counts only 1s
    n="count"
)

per["rate"] = per["k"] / per["n"]
per["N_h"] = sz["N_h"].reindex(per.index)
print(per.round(3))

kept = per.index.str.startswith("K_")                      # strata the filter kept
N = per["N_h"].to_numpy()

# point estimates: in-scope count per stratum = N_h * labeled rate
est = N * per["rate"].to_numpy()
p0 = est[kept].sum() / N[kept].sum()
r0 = est[kept].sum() / est.sum()
f0 = 2 * p0 * r0 / (p0 + r0)

# 95 % intervals: draw each stratum rate from Beta(k + 0.5, n - k + 0.5) (Jeffreys), so a 0/40 stratum is not treated as exactly zero
rng = np.random.default_rng(1)
rates = rng.beta(per["k"].to_numpy() + 0.5, (per["n"] - per["k"]).to_numpy() + 0.5, size=(4000, len(per)))
tp = (N * rates)[:, kept].sum(axis=1)
prec = tp / N[kept].sum()
rec = tp / (N * rates).sum(axis=1)
f1 = 2 * prec * rec / (prec + rec)
(lo_p, hi_p), (lo_r, hi_r), (lo_f, hi_f) = [np.percentile(x, [2.5, 97.5]) for x in (prec, rec, f1)]

print(f"\nFilter {FILTER_VERSION}  (audit frame excludes development-seen queries)")
print(f"precision : {p0:.3f}   95% interval [{lo_p:.3f}, {hi_p:.3f}]")
print(f"recall    : {r0:.3f}   95% interval [{lo_r:.3f}, {hi_r:.3f}]")
print(f"F1        : {f0:.3f}   95% interval [{lo_f:.3f}, {hi_f:.3f}]")

unsure / unlabeled: 12 of 310
                         k   n   rate    N_h
stratum                                     
D1_vetoed_high_share   5.0  37  0.135    100
D2_near_miss          19.0  45  0.422   1251
D3_low_share           3.0  30  0.100   3080
D4_bulk                0.0  40  0.000  72472
D5_vetoed_other        0.0  30  0.000  14601
K_A_ruleA             63.0  68  0.926   4038
K_B_ruleB             38.0  48  0.792   1407

Filter v2.3  (audit frame excludes development-seen queries)
precision : 0.892   95% interval [0.824, 0.934]
recall    : 0.851   95% interval [0.452, 0.864]
F1        : 0.871   95% interval [0.598, 0.882]


### Recorded results - Step 4 audit of filter v2.3 (310 blind, stratified, fresh queries; seed 2026)

| Stratum | N | Audited (labeled) | In-scope rate (Wilson 95%) |
|---|---:|---:|---|
| K_A Rule A (kept) | 4,038 | 68 | 92.6% |
| K_B Rule B (kept) | 1,407 | 48 | 79.2% |
| D1 vetoed, high electronic share | 100 | 37 | 13.5% |
| D2 near-miss (term & share 0.3–0.5, or no term & share 0.5–0.8) | 1,251 | 45 | 42.2% |
| D3 low share (0.1–0.3) | 3,080 | 30 | 10.0% |
| D4 bulk (share < 0.1, unvetoed) | 72,472 | 40 | 0% |
| D5 other vetoed | 14,601 | 30 | 0% |

- **Precision:** 0.892 (95% interval: 0.824–0.934)
- **Recall:** 0.851 (95% interval: 0.452–0.864)
- **F1:** 0.871 (95% interval: 0.598–0.882)
- **Uncertain labels:** 12 of 310 rows were labeled "unsure" or left unlabeled.

**Error structure.** False positives are mostly scope-boundary cases (car stereo / tweeter adapter, camera dolly, CPU water block, SSD enclosure, TV antenna, Apple Pencil, mic preamp, Roku) and a few non-electronics (`mac os book`, `polaroid book album`). Misses are mostly short generic queries just under the cutoffs (`work headphones`, `jbl powered speakers`, `outdoor fan`, `amazon echo`, `huawei t3 10`), device queries whose E/S lists are dominated by accessories (`apple watch 40mm series 5`), and accessory-vetoed queries that name a device (`earbuds with case for kids`).

## Step 5 - Freeze the dataset: final tables, retrieval corpus, dataset card
Decisions implemented here:
* **Queries** = the v2.3 kept set (US, Rule A + Rule B), with ESCI `split`, `small_version`, `source`, `scope_group`, `rule`, `share_es` carried along.
* **Judgments** = the **full** ESCI judged list of every kept query (E/S/C/I), with NDCG gains from the ESCI paper: **E 1.0 · S 0.1 · C 0.01 · I 0.0**. Unjudged products count as 0.
* **Retrieval corpus** = the pooled set of products judged for any kept query (train + test). Every query is answered against the whole pool, so the other queries' products act as natural distractors; non-electronic items that were judged *Irrelevant* for a kept query stay in as hard negatives.
* **Evaluation protocol:** classifier / prompt development on `split == "train"`; the ESCI `test` split is used once, for the final four-condition ablation. Report the full test set and the `small_version` test subset (ESCI's published ranking benchmark) separately.

In [7]:
GAIN = {"E": 1.0, "S": 0.1, "C": 0.01, "I": 0.0}

fq = q[q["v2_keep"]].copy()
fq["rule"] = np.where(fq["rule_A"], "A", "B")
queries_final = fq[["query", "split", "small", "source", "scope_group", "rule", "n_es", "share_es", "n_judged"]].reset_index()
queries_final["filter_version"] = FILTER_VERSION

judg = us_examples[us_examples["query_id"].isin(fq.index)][
    ["example_id", "query_id", "product_id", "esci_label", "split", "small_version", "large_version"]].copy()
judg["gain"] = judg["esci_label"].map(GAIN)
corpus = us_products[us_products["product_id"].isin(judg["product_id"].unique())].copy()

# Clean the two long text columns of the corpus (the title is left untouched)
for col in ["product_bullet_point", "product_description"]:
    corpus[col] = (corpus[col]
                   .map(html.unescape, na_action="ignore")      # &amp; -> &   &nbsp; -> space
                   .str.replace(r"<[^>]*>", " ", regex=True)    # remove tags: <p> </p> <strong> <br> ...
                   .str.replace(r"\\[nrt]", " ", regex=True)    # remove the text "\n" "\r" "\t" if it was saved as plain characters
                   .str.replace(r"\s+", " ", regex=True)        # real new lines, tabs and double spaces -> one space
                   .str.strip())                                # remove spaces at the start and end

# integrity checks
assert judg["product_id"].isin(corpus["product_id"]).all(), "judged product missing from corpus"
assert not judg.duplicated(["query_id", "product_id"]).any(), "duplicate (query, product) judgments"
assert judg["gain"].notna().all()

queries_final.to_parquet(OUT / f"esci_electronics_{FILTER_VERSION}_queries.parquet")
judg.to_parquet(OUT / f"esci_electronics_{FILTER_VERSION}_judgments.parquet")
corpus.to_parquet(OUT / f"esci_electronics_{FILTER_VERSION}_corpus.parquet")
card = {"dataset": "ESCI electronics subset (US)", "filter_version": FILTER_VERSION,
        "thresholds": {"rule_A_share": A_CUT, "rule_B_share": B_CUT, "min_es_products": N_MIN},
        "ndcg_gains": GAIN, "n_queries": int(len(queries_final)), "n_judgments": int(len(judg)), "n_corpus_products": int(len(corpus)),
        "audit": {"n": int(len(d0)), "seed": 2026, "precision": round(float(p0), 3), "recall": round(float(r0), 3)}}
json.dump(card, open(OUT / f"esci_electronics_{FILTER_VERSION}_card.json", "w"), indent=2)
print("queries:", len(queries_final), "| judgment rows:", len(judg), "| corpus products:", len(corpus), "| wrote 3 parquet files + card.json")

queries: 5664 | judgment rows: 100724 | corpus products: 51318 | wrote 3 parquet files + card.json


### Recorded results — Step 4 (final estimator) and Step 5 (frozen dataset v2.3)

**Audit (310 blind, stratified, unseen queries).** Precision **0.892** (95% interval 0.824–0.934). Recall point estimate **0.851**, with a 95% interval of **0.452–0.864**. The unvetoed bulk stratum D4 (\(N=72{,}472\)) produced 0/40 in-scope queries, so its true in-scope rate remains uncertain. Recall sensitivity to assumed D4 in-scope rates of 0%, 0.25%, 0.5%, and 1% should be recalculated using the updated final estimator before reporting. **12 of 310** rows were labeled "unsure" or left unlabeled. Conditional recall among queries with electronics evidence should be reported only after verifying the estimator's definition and calculation.

**Reporting sentence:** *Precision was 0.89 (95% interval 0.82–0.93), with an estimated recall of 0.85 (95% interval 0.45–0.86). Recall uncertainty is substantial because the large unvetoed bulk stratum contained no in-scope queries among the 40 audited.*

**Frozen dataset v2.3.** The export produced **5,664 queries**, **100,724 judgment rows**, and a pooled corpus of **51,318 products**. Three Parquet files and `card.json` were written successfully.

The dataset contains 5,664 queries and 100,724 query–product judgments, averaging approximately 17.8 judgments per query. The pooled product corpus contains 51,318 products. Previously recorded train/test split statistics, label distributions, product-text completeness, cross-split product overlap, and FAISS index size should be retained only after confirming they correspond to this updated export.

**Reproducibility note:** Filter v2.3 remains frozen. Do not retune it using the blind audit results. Recompute any derived statistics affected by the updated query and judgment counts before finalizing the research report.

## Step 6 - Classifier pool: stratified sample of 550 train queries to label

### Labeling protocol
**`intent`** - one of `Exact`, `Hybrid`, `Semantic`, `N/A`
1. **N/A** - not a device/component query in scope.
2. **Exact** - the query *pins down* one product (or one tiny family such as colour variants): a model / part code, or product line + number / generation, or a named brand with an exact spec combination.
3. **Hybrid** - a **brand or product-family anchor** is present but several different products fit (no pinning code); the keyword anchor and the descriptive words both matter. A misspelled brand still counts as a brand.
   `jbl speaker` · `adata ssd` · `amd ryzen` · `nikon zoom lens` · `kingston digital datatraveler flash drive` · `wireless earbuds samsung galaxy buds` · `nubwo gaming headset` · `sabrent 16 port usb hub` · `samsumg bluetooth headphones` · `I would buy a macbook pro … refurbished`
4. **Semantic** - **no** brand / product-line / model anchor: a category, attribute, use-case or need.
   `wireless home theater speakers` · `indoor security camera` · `karaoke machine for adults` · `lightning to hdmi cable` · `vibration sensor connect to phone` · `An everyday laptop for surfing the net and emails and not too expensive.`

**`price`** - `1` only if the query states a price or budget constraint (`under 100`, `$20`, `max budget £400`, `around £400`, `cheap`, `cheapest`, `budget`, `affordable`, `on sale`, `discount`), otherwise `0`. Independent of `intent`.

In [8]:
# 6A. heuristic cell of every train query (used for stratification only)
BRAND_ALLOWLIST = [
    "samsung", "apple", "xiaomi", "sony", "lg", "dell", "hp", "lenovo", "asus", "acer", "panasonic", "canon",
    "nikon", "gopro", "logitech", "jbl", "bose", "anker", "tp-link", "netgear", "corsair", "kingston",
    "western digital", "seagate", "oneplus", "oppo", "vivo", "realme", "huawei", "honor", "google", "motorola",
    "nokia", "tecno", "infinix", "poco", "redmi", "nothing", "fitbit", "garmin", "amazfit", "zte", "microsoft",
    "msi", "razer", "toshiba", "sandisk", "crucial", "adata", "gigabyte", "intel", "amd", "nvidia", "evga",
    "hyperx", "g.skill", "teamgroup", "pny", "wd", "sennheiser", "beats", "skullcandy", "jabra", "soundcore",
    "marshall", "harman kardon", "bang & olufsen", "ultimate ears", "edifier", "creative", "audio-technica",
    "ugreen", "baseus", "belkin", "aukey", "d-link", "linksys", "ubiquiti", "tenda", "mercusys", "cable matters",
    "amazon basics", "dji", "insta360", "arlo", "ring", "wyze", "eufy", "reolink", "hikvision", "dahua",
    "fujifilm", "olympus", "tcl", "hisense", "vizio", "sharp", "philips", "skyworth",
]
BRAND_RX = re.compile(r"\b(?:" + "|".join(re.escape(b) for b in BRAND_ALLOWLIST) + r")\b", re.I)
PRICE_RX = re.compile(r"(?:\bunder\b|\bbelow\b|\bless than\b|\bcheap\w*\b|\bbudget\b|\baffordable\b|\bdiscount\w*\b|\bon sale\b|[$£€]\s?\d|\b\d+\s?(?:usd|dollars?|bucks)\b)", re.I)
SPEC = re.compile(r"^\d+(?:\.\d+)?(?:gb|tb|mb|mah|w|hz|inch|in|ft|mm|cm|kg|lb|lbs|oz|ml|pack|pcs|ct|k|mp|v|a|ghz|fps)$", re.I)

pool = queries_final[queries_final["split"] == "train"].copy()
pool["n_words"] = pool["query"].str.split().str.len()
pool["has_price"] = [bool(PRICE_RX.search(t)) for t in pool["query"]]
# model token = a word with letters AND digits that is not a plain spec like "16gb"
pool["has_model"] = [any(re.search(r"[a-z]", w) and re.search(r"\d", w) and not SPEC.match(w)
                         for w in re.findall(r"[a-z0-9][a-z0-9-]{2,}", t.lower())) for t in pool["query"]]
pool["has_brand"] = [bool(BRAND_RX.search(t)) for t in pool["query"]]
pool["heuristic"] = np.select(
    [pool["has_price"], pool["has_model"], pool["has_brand"]],
    ["price (Hybrid-like)", "model token (Exact-like)", "brand only (mixed)"], default="neither (Semantic-like)")

# 6B. stratified draw: all nlqec + all price queries; the rest by heuristic cell and length bucket (weights 1 / 1 / 1.5 / 2)
rng = np.random.default_rng(2026)
pool["bucket"] = pd.cut(pool["n_words"], [0, 2, 4, 6, 1000], labels=["1-2", "3-4", "5-6", "7+"]).astype(str)
pool["stratum6"] = np.select([pool["source"] == "nlqec", pool["has_price"], pool["has_model"], pool["has_brand"]],
                             ["NL_all", "price_all", "model", "brand_only"], default="neither")
QUOTA = {"NL_all": None, "price_all": None, "model": 110, "brand_only": 150, "neither": 188}   # None = take all
BUCKET_W = pd.Series({"1-2": 1.0, "3-4": 1.0, "5-6": 1.5, "7+": 2.0})

picked = []
for st, n in QUOTA.items():
    df = pool[pool["stratum6"] == st]
    if n is None or n >= len(df):
        picked += list(df.index)
        continue
    cnt = df["bucket"].value_counts().reindex(BUCKET_W.index).fillna(0)
    w = cnt * BUCKET_W
    alloc = np.minimum((w / w.sum() * n).round().astype(int), cnt.astype(int))
    chosen = []
    for b, k in alloc.items():
        ids = df.index[df["bucket"] == b].to_numpy()
        chosen += list(rng.choice(ids, size=int(k), replace=False))
    if len(chosen) < n:                                    # top up from whatever is left
        rest = df.index.difference(chosen).to_numpy()
        chosen += list(rng.choice(rest, size=min(n - len(chosen), len(rest)), replace=False))
    picked += chosen[:n]
samp = pool.loc[picked].copy()

title_of = us_products.set_index("product_id")["product_title"]
rows = us_examples[us_examples["query_id"].isin(samp["query_id"]) & us_examples["esci_label"].isin(["E", "S"])].sort_values(["query_id", "esci_label"])
ex_titles = rows.groupby("query_id")["product_id"].apply(lambda s: " || ".join(title_of.reindex(s.iloc[:2]).fillna("").str.slice(0, 90)))
samp["example_titles"] = samp["query_id"].map(ex_titles).fillna("")

blind = samp.sample(frac=1, random_state=11).reset_index(drop=True)
blind["label_id"] = np.arange(1, len(blind) + 1)
blind[["label_id", "query", "example_titles"]].assign(intent="", price="", note="").to_csv(OUT / "classifier_pool_to_label.csv", index=False)
blind[["label_id", "query_id", "stratum6", "bucket", "heuristic", "source", "scope_group"]].to_csv(OUT / "classifier_pool_key.csv", index=False)
print(len(blind), "queries to label\n"); print(pd.crosstab(blind["stratum6"], blind["bucket"], margins=True))

554 queries to label

bucket      1-2  3-4  5-6  7+  All
stratum6                          
NL_all        0    0    0  74   74
brand_only   29   78   37   6  150
model        40   45   18   7  110
neither      47   91   39  11  188
price_all     4   14   13   1   32
All         120  228  107  99  554


In [9]:
# 6C. RUN AFTER LABELING: classifier files, 70 / 15 / 15 split stratified on `intent`
# keep_default_na=False: otherwise pandas turns the label "N/A" (and "NA") into a missing value and silently drops those rows
lab = pd.read_csv(OUT / "classifier_pool_labeled.csv", dtype=str, keep_default_na=False)
lab["label_id"] = lab["label_id"].astype(int)
d = lab.merge(pd.read_csv(OUT / "classifier_pool_key.csv"), on="label_id")
d["intent"] = d["intent"].astype(str).str.strip().str.lower().map(
    {"exact": "Exact", "semantic": "Semantic", "hybrid": "Hybrid", "na": "N/A", "n/a": "N/A"})
d["price"] = pd.to_numeric(d["price"], errors="coerce").fillna(0).astype(int)
d = d.dropna(subset=["intent"])
print("intent counts:", d["intent"].value_counts().to_dict())

use = d[d["intent"].isin(["Exact", "Semantic", "Hybrid"])].copy()       # N/A rows are reported above, not used
print("usable labeled queries:", len(use), "(need >= 500)")

rng = np.random.default_rng(42)
tr, va, te = [], [], []
for _, g in use.groupby("intent"):
    idx = rng.permutation(g.index.to_numpy()); n = len(g)
    a, b = int(round(0.70 * n)), int(round((0.70 + 0.15) * n))
    tr += list(idx[:a]); va += list(idx[a:b]); te += list(idx[b:])

for name, rows_ in [("train", tr), ("val", va), ("test", te)]:
    part = use.loc[rows_]
    part[["query_id", "query", "intent", "price"]].to_csv(f"classifier_{name}.csv", index=False)
    print(f"{name}: {len(part)}", part["intent"].value_counts().to_dict())

intent counts: {'Hybrid': 216, 'Semantic': 204, 'Exact': 132, 'N/A': 2}
usable labeled queries: 552 (need >= 500)
train: 386 {'Hybrid': 151, 'Semantic': 143, 'Exact': 92}
val: 83 {'Hybrid': 33, 'Semantic': 30, 'Exact': 20}
test: 83 {'Hybrid': 32, 'Semantic': 31, 'Exact': 20}


### Step 7 - Retrieval-ready exports (added after the v2.3 freeze; filter unchanged)

In [10]:
def make_doc(title, brand, color, bullets, n_bullet=600):
    bullets = bullets if isinstance(bullets, str) else ""
    parts = [title, brand, color, bullets[:n_bullet]]
    return " | ".join(p.strip() for p in parts if isinstance(p, str) and p.strip())

corpus_r = corpus.copy()
corpus_r["doc_text"] = [make_doc(t, b, c, bp) for t, b, c, bp in zip(
    corpus_r["product_title"], corpus_r["product_brand"],
    corpus_r["product_color"], corpus_r["product_bullet_point"])]

corpus_r = corpus_r[["product_id", "product_title", "product_brand", "product_color",
                     "product_bullet_point", "product_description", "doc_text"]].reset_index(drop=True)
corpus_r.to_parquet(OUT / f"esci_electronics_{FILTER_VERSION}_corpus_retrieval.parquet")
print(corpus_r.shape, "| empty doc_text:", int((corpus_r["doc_text"] == "").sum()))
print(corpus_r["doc_text"].str.split().str.len().describe().round(1))
print(corpus_r["doc_text"].iloc[0])

(51318, 7) | empty doc_text: 0
count    51318.0
mean        99.1
std         38.0
min          1.0
25%         74.0
50%        115.0
75%        127.0
max        199.0
Name: doc_text, dtype: float64
Samsung ET-YO324BBEGUS Gear VR Controller | SAMSUNG | Black | Motion controller for more realistic interaction Ability to recognize hand motion, evolving interactions and gameplay in VR An ergonomic design with a trigger for natural, intuitive control Unlocks even more amazing experiences in VR.Compatible OS Android Lollipop 5.0 or later The Gear VR controller works with Galaxy S8 and Galaxy S8+ as well as previous VR-enabled handsets, including Galaxy S7, Galaxy S7 edge, Galaxy S6, Galaxy S6 edge, Galaxy S6 edge+ and Galaxy Note5 and previous generation Gear VR headsets.


In [11]:
clf = {n: pd.read_csv(f"classifier_{n}.csv") for n in ["train", "val", "test"]}
for n, d_ in clf.items():
    d_.to_csv(OUT / f"classifier_{n}.csv", index=False)

qf = queries_final.copy()
qf["role"] = np.where(qf["split"] == "test", "eval_test", "train_other")
for n, d_ in clf.items():
    qf.loc[qf["query_id"].isin(d_["query_id"]), "role"] = f"clf_{n}"

rng = np.random.default_rng(2026)
cand = qf.index[qf["role"] == "train_other"].to_numpy()
dev_idx = rng.choice(cand, size=min(500, len(cand)), replace=False)
qf.loc[dev_idx, "role"] = "dev"

qf[["query_id", "role"]].to_csv(OUT / f"query_roles_{FILTER_VERSION}.csv", index=False)
print(qf["role"].value_counts())
print(pd.crosstab(qf["role"], qf["small"]))

role
train_other    3317
eval_test      1295
dev             500
clf_train       386
clf_test         83
clf_val          83
Name: count, dtype: int64
small           0    1
role                  
clf_test       58   25
clf_train     271  115
clf_val        64   19
dev           382  118
eval_test     788  507
train_other  2405  912


In [12]:
has_E = judg.groupby("query_id")["esci_label"].apply(lambda s: (s == "E").any()).rename("has_E")
chk = qf.merge(has_E, left_on="query_id", right_index=True, how="left")
print("queries with no Exact product, by role:")
print(chk.groupby("role")["has_E"].apply(lambda s: int((~s).sum())))
print("\njudged products per query, by role:")
print(judg.merge(qf[["query_id", "role"]], on="query_id").groupby("role").size()
      / qf["role"].value_counts())
print("\nproducts judged for >1 query:", int((judg["product_id"].value_counts() > 1).sum()))

queries with no Exact product, by role:
role
clf_test       0
clf_train      0
clf_val        0
dev            0
eval_test      0
train_other    0
Name: has_E, dtype: int64

judged products per query, by role:
role
clf_test       16.867470
clf_train      17.321244
clf_val        16.638554
dev            18.404000
eval_test      17.874131
train_other    17.759421
dtype: float64

products judged for >1 query: 15902
